# Vilier Colab Runner

Notebook nay mount Google Drive, clone/pull repo, cai dependencies, chay `bash run.sh` voi input lay tu Drive va output ghi lai Drive.

In [1]:
# Sua cac gia tri nay truoc khi chay neu can.
REPO_URL = "https://github.com/ngocbao220/vilier.git"
BRANCH = "main"
PROJECT_DIR = "/content/vilier"

# Dat file audio trong Google Drive, vi du: MyDrive/VDT-TurnTaking/inputs/real.wav
DRIVE_AUDIO_PATH = "/content/drive/MyDrive/VDT-TurnTaking/inputs/real.wav"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/VDT-TurnTaking/outputs"

# Colab GPU: Runtime -> Change runtime type -> T4 GPU.
PIPELINE_DEVICE = "gpu"

# Diarization local tren Colab bang pyannote.audio 4.x community-1.
DIARIZATION_BACKEND = "pyannote"
DIARIZATION_MODEL = "pyannote/speaker-diarization-community-1"

# Bat ASR neu muon chay PhoWhisper sau khi tach speaker.
ENABLE_ASR = False
ENABLE_STATE_LABELING = False
DRY_RUN = False


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import os
import subprocess
from pathlib import Path

project_dir = Path(PROJECT_DIR)
if project_dir.exists():
    subprocess.run(["git", "-C", str(project_dir), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(project_dir), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(project_dir), "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(project_dir)], check=True)

os.chdir(project_dir)
print("Repo:", project_dir)


Repo: /content/vilier


In [4]:
import sys
import subprocess

subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)

# community-1 can pyannote.audio >= 4.0. Cai sau requirements vi requirements cua repo co the pin ban 3.x cho PixIT.
subprocess.run([sys.executable, "-m", "pip", "install", "-U", "pyannote.audio", "torchvision"], check=True)

subprocess.run(
    [
        sys.executable,
        "-c",
        "import torch, torchvision, pyannote.audio; "
        "print('torch', torch.__version__); "
        "print('torchvision', torchvision.__version__); "
        "print('pyannote.audio', pyannote.audio.__version__); "
        "print('cuda_available', torch.cuda.is_available())",
    ],
    check=True,
)


CalledProcessError: Command '['/usr/bin/python3', '-c', "import torch, torchvision, pyannote.audio; print('torch', torch.__version__); print('torchvision', torchvision.__version__); print('pyannote.audio', pyannote.audio.__version__); print('cuda_available', torch.cuda.is_available())"]' returned non-zero exit status 1.

In [5]:
import json
import os
import sys
from pathlib import Path

try:
    from google.colab import userdata
except Exception:
    userdata = None

def set_secret_from_colab(secret_name, env_name):
    if os.environ.get(env_name) or userdata is None:
        return
    try:
        value = userdata.get(secret_name)
    except Exception:
        value = None
    if value:
        os.environ[env_name] = value

set_secret_from_colab("HUGGINGFACE_TOKEN", "HUGGINGFACE_TOKEN")
set_secret_from_colab("HF_TOKEN", "HF_TOKEN")
set_secret_from_colab("DASHSCOPE_API_KEY", "DASHSCOPE_API_KEY")

if not os.environ.get("HUGGINGFACE_TOKEN") and not os.environ.get("HF_TOKEN"):
    raise RuntimeError("Set HUGGINGFACE_TOKEN in Colab secrets or os.environ before running pyannote community-1.")

audio_path = Path(DRIVE_AUDIO_PATH)
output_dir = Path(DRIVE_OUTPUT_DIR)
if not audio_path.exists():
    raise FileNotFoundError(f"Drive audio not found: {audio_path}")
output_dir.mkdir(parents=True, exist_ok=True)

with open("config.json", "r", encoding="utf-8") as handle:
    config = json.load(handle)

config.setdefault("entrypoint", {})["input_path"] = str(audio_path)
config.setdefault("entrypoint", {})["output_path"] = str(output_dir)
config.setdefault("runtime", {})["dry_run"] = bool(DRY_RUN)
config.setdefault("logging", {})["progress_bar"] = True

# Guard against stale Colab variables from older cells where backend was accidentally set to the model id.
diarization_backend = str(DIARIZATION_BACKEND).strip()
diarization_model = str(DIARIZATION_MODEL).strip()
if "/" in diarization_backend and diarization_backend.startswith("pyannote"):
    diarization_model = diarization_backend
    diarization_backend = "pyannote"

config["diarization"] = {
    "backend": diarization_backend,
    "model": diarization_model,
    "token_env": "HUGGINGFACE_TOKEN",
    "device": PIPELINE_DEVICE,
    "min_duration_seconds": 0.25,
    "max_chunk_seconds": 180.0,
}

config.setdefault("asr", {})["enabled"] = bool(ENABLE_ASR)
config["asr"]["device"] = PIPELINE_DEVICE
config.setdefault("state_labeling", {})["enabled"] = bool(ENABLE_STATE_LABELING)

config.setdefault("music_separation", {})["enabled"] = False
config["music_separation"]["device"] = PIPELINE_DEVICE
config.setdefault("overlap_separation", {})["enabled"] = False
config["overlap_separation"]["device"] = PIPELINE_DEVICE

config_path = Path("config.colab.json")
config_path.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")

os.environ["CONFIG_PATH"] = str(config_path)
os.environ["INPUT_PATH"] = str(audio_path)
os.environ["OUTPUT_PATH"] = str(output_dir)
os.environ["PYTHON_BIN"] = sys.executable
if DRY_RUN:
    os.environ["DRY_RUN"] = "1"

print("CONFIG_PATH=", os.environ["CONFIG_PATH"])
print("INPUT_PATH=", os.environ["INPUT_PATH"])
print("OUTPUT_PATH=", os.environ["OUTPUT_PATH"])
print("DIARIZATION_BACKEND=", diarization_backend)
print("DIARIZATION_MODEL=", diarization_model)
print("PIPELINE_DEVICE=", PIPELINE_DEVICE)
print("ENABLE_ASR=", ENABLE_ASR)
print("ENABLE_STATE_LABELING=", ENABLE_STATE_LABELING)


RuntimeError: Set HUGGINGFACE_TOKEN in Colab secrets or os.environ before running pyannote community-1.

In [6]:
!bash run.sh


[INFO] Pipeline component usage
| Component          | Enabled | Backend          | Model                                    |
| ------------------ | ------- | ---------------- | ---------------------------------------- |
| VAD                | V       | silero           | silero_vad                               |
| Diarization        | V       | pyannote         | pyannote/speaker-diarization-community-1 |
| Music separation   | X       | demucs           | htdemucs                                 |
| Overlap separation | X       | sepreformer      | SepReformer_Large_DM_WHAMR               |
| ASR                | X       | phowhisper_local | vinai/PhoWhisper-large                   |
| State labeling     | X       | qwen             | qwen3.8-max                              |
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/vilier/pipeline/cli.py", line 764, in <module>
    

In [ ]:
from pathlib import Path

audio_id = Path(DRIVE_AUDIO_PATH).stem
result_dir = Path(DRIVE_OUTPUT_DIR) / audio_id
print("Result dir:", result_dir)
for path in sorted(result_dir.rglob("*")):
    if path.is_file():
        print(path.relative_to(result_dir))
